# 01 - Exploratory Data Analysis (EDA)

This notebook performs an exploratory analysis of the **Bank Telemarketing** dataset.  
The main goals are to:

- Understand the **structure and quality** of the data
- Explore **univariate** and **bivariate** patterns
- Investigate **relationships** between predictors and the subscription target
- Identify **high-impact features** and **potential data issues** (skewness, imbalance, leakage)
- Generate **feature engineering** ideas for the modeling phase

This EDA focuses on:
- Customer demographics  
- Campaign behavior and history  
- Macroeconomic indicators  
- Target conversion behavior (subscribed vs. not subscribed)

>**Important:** The `duration` feature will be treated as a *leakage variable* and **must not be used** in any realistic predictive model, since it is only known *after* the call.

## 1. Imports, Configuration & Plot Styling

In this section the following activities are done:

- Import shared utilities from `utils.py`
- Define a consistent **color theme** for all plots
- Create a timestamped folder to store generated figures

The environment variables `BRONZE_PATH` and `FIG_DIR` come from `.env` and are loaded in `utils.py`, ensuring a clean separation between code and configuration.

In [ ]:
! pip install -q -r ../requirements.txt

In [ ]:
# --- Basic Imports ---
from utils import *

# Plot Themes
ACCENT_COLOR = "#2ab7ca"
SECOND_COLOR = "#0d3b66"
HIGHLIGHT_COLOR = "#ff0000ea"

fig_folder_time = dt.datetime.now().strftime("%Y%m%d-%H%M%S")

# Plot Download Folder Path
fig_download_path = f"{FIG_DIR}{fig_folder_time}"

# --- Theme & custom accent color ---
sns.set_theme(style="whitegrid", palette="mako")

# --- Create Figures Directory ---
os.makedirs(fig_download_path, exist_ok=True)


## 2. Dataset Exploration

### 2.1 Load Dataset & Initial Overview

We start by loading the **bronze-layer** dataset and normalizing column names using `normalizeString` from `utils.py`.

Key questions at this stage:

- Did the dataset load correctly?
- How many rows and columns do we have?
- Which features are **numeric** vs **categorical**?
- Are there obvious **missing values** or `"unknown"` categories?
- Does the schema match the project description?

We then inspect:

- `.head()` – first rows for a quick data check
- `.info()` – data types and null counts
- `.describe()` – basic distribution statistics for numeric and categorical variables

In [ ]:
# --- Load Dataset ---
df = pd.read_csv(f"{BRONZE_PATH}bank-additional-full.csv", sep=";")
df.columns = [normalizeString(colname) for colname in df.columns]
df.head()

### 2.2 Feature Groups

The dataset contains customer demographics, campaign details, and macroeconomic indicators.  
For clarity, we group features as follows:

**Customer Demographics**
- `age` — Customer age  
- `job` — Type of job
- `marital` — Marital status
- `education` — Highest education level
- `default` — Credit in default?
- `housing` — Has a housing loan?
- `loan` — Has a personal loan?

**Current Campaign Information**
- `contact` — Contact communication type
- `month` — Month of last contact
- `day_of_week` — Day of last contact
- `duration` — Call duration in seconds

**Campaign History**
- `campaign` — Number of contacts during the current campaign
- `pdays` — Days since last contact in previous campaign
- `previous` — Number of prior contacts
- `poutcome` — Outcome of previous campaign

**Economic Indicators**
- `emp_var_rate` — Employment variation rate
- `cons_price_idx` — Consumer price index
- `cons_conf_idx` — Consumer confidence index
- `euribor3m` — 3-month Euribor rate
- `nr_employed` — Number of employees

**Target Variable**
- `y` — Whether the client subscribed to a term deposit (`yes`/`no`)

**Note:** `duration` feature should not be used for predictions due to leakage. This field indicates the call duration, so it is information that we only have after the call has been done, so it wouldn't be available at prediction time.


In [ ]:
df.info()
df.describe(include="all")

### 2.3 Numerical vs. Categorical Features

We now separate columns into **numerical** and **categorical** variables.  
This helps us:

- Choose appropriate visualizations (histograms vs. countplots)
- Plan preprocessing (scaling vs encoding)
- Structure the EDA and later ML pipelines more cleanly

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

num_cols, cat_cols

## 3. Feature Exploration

### 3.1 Univariate Analysis – Numerical Features

We visualize the distribution of each numerical variable using histograms with KDE curves.

Things to watch for:

- **Skewness** (e.g., `campaign`, `pdays`, `previous`)
- **Outliers** (extremely large values)
- Variables with very **low variability** (potentially low information)

In [ ]:
n = len(num_cols)
cols = 3
rows = (n // cols) + (n % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color=ACCENT_COLOR)
    axes[i].set_title(f"Distribution of {col}")

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Numerical Feature Distributions", fontsize=16)
plt.tight_layout()
plt.savefig(f"{fig_download_path}/all_numeric_histograms.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.2 Boxplots – Numerical Variables

Boxplots summarize central tendency, spread, and potential outliers for each numeric feature.

We use them to:

- Confirm the presence of **extreme values**
- Compare the spread of different variables
- Flag features that might require transformations or more complex modeling approaches

In [ ]:
n = len(num_cols)
cols = 3
rows = (n // cols) + (n % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i], color=ACCENT_COLOR)
    axes[i].set_title(f"Boxplot — {col}")

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Numerical Feature Boxplots", fontsize=16)
plt.tight_layout()
plt.savefig(f"{fig_download_path}/all_numeric_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()

### 3.3 Categorical Feature Distributions

We examine the distribution of each categorical variable (job, marital status, education, contact type, etc.) to understand:

- Dominant customer segments
- Rare or underrepresented categories
- `"unknown"` categories that may need special treatment (e.g., keep as separate level vs impute)

In [ ]:
n_cat = len(cat_cols)
cols = 2
rows = (n_cat // cols) + int(n_cat % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    # Order categories by frequency
    order = df[col].value_counts().index
    
    sns.countplot(
        data=df,
        x=col,
        order=order,
        ax=ax,
        color=ACCENT_COLOR
    )
    ax.set_title(f"Value Counts — {col}")
    ax.tick_params(axis="x", rotation=45)

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Categorical Feature Distributions", fontsize=18)
plt.tight_layout()
plt.savefig(f"{fig_download_path}/all_categorical_value_counts.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. Bivariate Analysis

### 4.1 Success Rate by Categorical Features

We create barplots of the mean subscription rate (`y_bin`) for each category of all categorical variables.

This helps identify:

- **High-conversion** categories (e.g., specific jobs, months, contact types)
- **Low-conversion** categories
- Potential **segmentation opportunities** for marketing

In [ ]:
df["y_bin"] = df["y"].map({"yes":1, "no":0})

In [ ]:
n = len(cat_cols)
cols = 2
rows = (n // cols) + (n % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.barplot(data=df, x=col, y="y_bin", ax=axes[i], color=ACCENT_COLOR)
    axes[i].set_title(f"Success Rate — {col}")
    axes[i].tick_params(axis='x', rotation=45)

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Success Rate by Categorical Features", fontsize=18)
plt.tight_layout()
plt.savefig(f"{fig_download_path}/all_categorical_success_rates.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.2 Numerical Features vs Target

We compare the distribution of numerical variables across the two target classes (`y` = yes/no) using boxplots.

We are looking for:

- Clear differences between subscribed vs. non-subscribed customers
- Variables that show **little separation** (likely weaker predictors)
- Outliers that might negatively impact model performance

In [ ]:
n = len(num_cols)
cols = 3
rows = (n // cols) + (n % cols > 0)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x="y", y=col, ax=axes[i], color=ACCENT_COLOR)
    axes[i].set_title(f"{col} vs Target")

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Numerical Variables vs Target", fontsize=18)
plt.tight_layout()
plt.savefig(f"{fig_download_path}/all_numeric_vs_target.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.3 Correlation Heatmap – Numerical Variables

We compute the Pearson correlation matrix for all numerical features and highlight strong correlations (|r| > 0.7).

This allows us to:

- Detect **multicollinearity** (especially among economic indicators)
- Identify potentially redundant variables
- Better understand relationships between campaign, customer, and macro features

In [ ]:
corr = df[num_cols].corr()

# Highlight: |corr| > 0.7 AND corr != 1
highlight_mask = corr.abs() > 0.7

# Prevent self-correlation highlight
np.fill_diagonal(highlight_mask.values, False)   

plt.figure(figsize=(14,10))

sns.heatmap(
    corr,
    cmap="mako",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar=True,
    annot_kws={"size": 9}
)

# Overlap Highlight boxes
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        if highlight_mask.iloc[i, j]:
            plt.gca().add_patch(
                plt.Rectangle(
                    (j, i), 1, 1,
                    fill=False,
                    edgecolor="#ff0000",
                    linewidth=2.5
                )
            )

plt.title("Pearson Correlation Heatmap — Highlighting Strong Correlations (>|0.7|)", fontsize=14)

plt.savefig(f"{fig_download_path}/correlation_heatmap_highlighted.png", dpi=300, bbox_inches="tight")
plt.show()


### 4.4 Cramér’s V – Categorical Associations

For categorical variables, we use Cramér’s V to measure association strength (0 = none, 1 = strong).

We highlight pairs with V > 0.30, which may indicate:

- Overlapping or redundant information
- Meaningful business relationships (e.g., job ↔ education)
- Strong interactions relevant for modeling

In [ ]:
# Function to compute Cramér's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2_corr = max(0, phi2 - (k-1)*(r-1)/(n-1))
    r_corr = r - (r-1)**2/(n-1)
    k_corr = k - (k-1)**2/(n-1)
    return np.sqrt(phi2_corr / min((k_corr-1), (r_corr-1)))

# Compute Cramér's V matrix
cat_corr = pd.DataFrame(
    np.zeros((len(cat_cols), len(cat_cols))),
    index=cat_cols,
    columns=cat_cols
)

for col1 in cat_cols:
    for col2 in cat_cols:
        cat_corr.loc[col1, col2] = cramers_v(df[col1], df[col2])

# Highlight: |corr| > 0.3 AND corr != 1
highlight_mask = cat_corr.abs() > 0.30

# Prevent self-correlation highlight
np.fill_diagonal(highlight_mask.values, False)

plt.figure(figsize=(14,10))

sns.heatmap(
    cat_corr,
    cmap="mako",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 9},
    cbar=True
)

# Overlap Highlight boxes
for i in range(cat_corr.shape[0]):
    for j in range(cat_corr.shape[1]):
        if highlight_mask.iloc[i, j]:
            plt.gca().add_patch(
                plt.Rectangle(
                    (j, i), 1, 1,
                    fill=False,
                    edgecolor="#ff0000ea",  # your greenish-blue highlight
                    linewidth=2.5
                )
            )

plt.title("Cramér's V Heatmap — Categorical Association Strength", fontsize=14)

plt.savefig(f"{fig_download_path}/categorical_cramersV_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Segment Analysis

### 5.1 Age Group Performance

We segment customers into age bands (Young, Adult, Middle Age, Senior) and compare subscription rates across groups.

This helps answer:

- Which age segments are most likely to subscribe?
- Should age be modeled as **continuous** or **categorical**?
- Are there obvious targetable segments for the business?

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 40, 60, 100],
    labels=["Young", "Adult", "Middle Age", "Senior"]
)

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(data=df, x="age_group", y="y_bin", color=ACCENT_COLOR)
plt.title("Success Rate by Age Group")

plt.savefig(f"{fig_download_path}/age_group_success_rate.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.2 Success Rate by Job Category

We analyze conversion rates by job type to identify professional groups with:

- High responsiveness (e.g., students, retirees)
- Low responsiveness (e.g., certain blue-collar roles)

This is key for tailoring contact strategies and sales scripts.

In [ ]:
plt.figure(figsize=(10,4))
sns.barplot(data=df, x="job", y="y_bin", color=ACCENT_COLOR)
plt.xticks(rotation=45)
plt.title("Success Rate by Job")

plt.savefig(f"{fig_download_path}/job_success_rate.png", dpi=300, bbox_inches="tight")
plt.show()


### 5.3 Success Rate by Month

We investigate whether subscription success rates vary across months.

We’re interested in:

- Strong months vs weak months
- Potential **seasonality** patterns
- Whether month should be used as a categorical variable or encoded cyclically

In [ ]:
plt.figure(figsize=(10,4))
sns.barplot(
    data=df, x="month", y="y_bin",
    order=["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"],
    color=ACCENT_COLOR
)
plt.title("Success Rate by Month")

plt.savefig(f"{fig_download_path}/month_success_rate.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Economic Indicators

### 6.1 Distributions by Target

In this section we compare the distributions of macroeconomic indicators for subscribed vs. non-subscribed customers:

- `emp_var_rate`
- `cons_price_idx`
- `cons_conf_idx`
- `euribor3m`
- `nr_employed`

By plotting these variables with the target (`y`), we want to understand:

- How **economic context** (interest rates, employment, confidence) affects campaign success
- Whether there are clear regions of the economic space where subscriptions are more likely
- Which indicators are most promising as predictive features

In [ ]:
for col in ["emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed"]:
    plt.figure(figsize=(8,4))

    sns.histplot(
        data=df, x=col, hue="y",
        palette=[ACCENT_COLOR, "#0d3b66"],
        kde=True, element="step"
    )

    plt.title(f"{col} Distribution by Target")

    plt.savefig(f"{fig_download_path}/{col}_vs_target_dist.png", dpi=300, bbox_inches="tight")
    plt.show()


## 7. Pairwise Relationships

### 7.1 Pairplots – Key Numerical Variables

Here we use a pairplot with the most relevant numerical variables (`age`, `campaign`, `pdays`, `euribor3m`, `emp_var_rate`) with the binary target (`y_bin`).

The pairplot helps us:

- Visualize relationships between pairs of features
- See whether **clusters or separations** appear for the subscribed vs. non-subscribed classes
- Spot obvious outliers or unusual regions in the feature space

In [ ]:
pairplot_cols = [
    "age",
    "campaign",
    "pdays",
    "euribor3m",
    "emp_var_rate",
    "y_bin"
]

g = sns.pairplot(
    df[pairplot_cols],
    hue="y_bin",
    palette={0: ACCENT_COLOR, 1: SECOND_COLOR},
    diag_kind="kde",
    corner=False
)

g.figure.suptitle("Pairplot — Key Numerical Variables", y=1.02)

plt.savefig(f"{fig_download_path}/pairplot_numeric.png", dpi=300, bbox_inches="tight")
plt.show()


### 7.2 Pairplots – Economic Indicators

We then focus specifically on the macroeconomic indicators:

- `euribor3m`
- `emp_var_rate`
- `cons_price_idx`
- `cons_conf_idx`
- `nr_employed`

The goal is to:

- Understand how these indicators move together over time
- Check whether specific economic conditions are associated with higher subscription rates
- Confirm the strong **multicollinearity** expected among these variables

In [ ]:
econ_cols = [
    "euribor3m",
    "emp_var_rate",
    "cons_price_idx",
    "cons_conf_idx",
    "nr_employed",
    "y_bin"
]

g = sns.pairplot(
    df[econ_cols],
    hue="y_bin",
    palette={0: ACCENT_COLOR, 1: SECOND_COLOR},
    diag_kind="kde",
    corner=False
)

g.figure.suptitle("Pairplot — Economic Indicators", y=1.02)

plt.savefig(f"{fig_download_path}/pairplot_economic.png", dpi=300, bbox_inches="tight")
plt.show()


### 7.3 PairGrid with Regression Lines

Using a `PairGrid`, we draw regression lines between numeric feature pairs (e.g. `age`, `campaign`, `pdays`, `euribor3m`, `emp_var_rate`).

This allows us to:

- Inspect potential **linear trends** between features
- See whether those trends differ by target class (`y_bin`)
- Validate whether linear models can capture most of the structure, or if relationships are mostly nonlinear

In [ ]:
reg_cols = [
    "age",
    "campaign",
    "pdays",
    "euribor3m",
    "emp_var_rate"
]

g = sns.PairGrid(df[reg_cols + ["y_bin"]], hue="y_bin")
g.map_lower(sns.regplot,
            scatter_kws={"alpha": 0.6},
            line_kws={"color": SECOND_COLOR})
g.map_diag(sns.kdeplot)
g.map_upper(sns.scatterplot, alpha=0.6,
            palette={0: ACCENT_COLOR, 1: SECOND_COLOR})

g.add_legend()
g.figure.suptitle("PairGrid with Regression Lines", y=1.02)

plt.savefig(f"{fig_download_path}/pairgrid_regression.png", dpi=300, bbox_inches="tight")
plt.show()


### 7.4 Mixed-Type PairGrid – Numerical & Categorical

Next, we build a mixed-type `PairGrid` combining:

- Encoded categorical features (`job`, `marital`, `education`)
- Numerical features (`age`, `campaign`, `pdays`)

With a mix of scatter, strip, and KDE plots, this view helps us:

- See how categorical groups differ across numeric ranges
- Identify categories (e.g., specific jobs or education levels) that cluster in distinct numeric regions
- Assess whether certain combinations of category + numeric features might be especially predictive

In [ ]:
# Choose categorical and numeric features
cat_subset = ["job", "marital", "education"]
num_subset = ["age", "campaign", "pdays"]

# Encode categories to integers for plotting
df_coded = df.copy()
for col in cat_subset:
    df_coded[col + "_code"] = df_coded[col].astype("category").cat.codes

mix_cols = [col + "_code" for col in cat_subset] + num_subset

# Create PairGrid for all mixed variables
g = sns.PairGrid(df_coded[mix_cols + ["y_bin"]], hue="y_bin",
                 palette={0: ACCENT_COLOR, 1: SECOND_COLOR})

# Lower triangle: scatter plots for numeric pairs
g.map_lower(sns.scatterplot, alpha=0.6)

# Diagonal: KDE for all types
g.map_diag(sns.kdeplot)

# Upper triangle: strip plots for categorical–numeric & categorical-categorical
g.map_upper(sns.stripplot, dodge=True, alpha=0.5)

g.figure.suptitle("Mixed-Type PairGrid — Categorical + Numerical", y=1.02)

plt.savefig(f"{fig_download_path}/pairgrid_mixed.png", dpi=300, bbox_inches="tight")
plt.show()

### 7.5 Pairplot – Separation by Target

Finally, we generate another pairplot focused on **separation between target classes** using a subset of key numerical features.

Here we are specifically looking for:

- Variables where subscribed and non-subscribed customers occupy clearly different regions
- Features that show heavy overlap between classes (weaker stand-alone predictors)
- Visual support for the importance of variables like `campaign`, `pdays`, and economic indicators

In [ ]:
target_pairplot_cols = [
    "age", "campaign", "pdays", "euribor3m", "emp_var_rate"
]

g = sns.pairplot(
    df[target_pairplot_cols + ["y_bin"]],
    hue="y_bin",
    kind="scatter",
    palette={0: ACCENT_COLOR, 1: SECOND_COLOR},
    diag_kind="kde",
    corner=False
)

g.figure.suptitle("Pairplot — Separation by Target", y=1.02)

plt.savefig(f"{fig_download_path}/pairplot_target.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Detailed EDA Interpretation (Analysis Notes)

The following notes summarize the main insights from the plots above.  
They are intended as a bridge between the visual EDA and the modeling recommendations» that follow.

### 8.1 Economic Indicator Distributions by Target

These plots compare key macroeconomic indicators (`euribor3m`, `emp_var_rate`, `cons_price_idx`, `cons_conf_idx`, `nr_employed`) between subscribed (`y = 1`) and non-subscribed (`y = 0`) customers.

**Key observations:**

- **Low Euribor → Higher Subscription Rates**  
  Subscriptions are more frequent when `euribor3m` is low, suggesting that in low interest rate environments customers are more open to term deposits.

- **Negative Employment Variation Encourages Subscriptions**  
  More subscriptions occur when `emp_var_rate` is negative, i.e., during periods of declining employment.

- **Consumer Price & Confidence Indices Have Weak Separation**  
  `cons_price_idx` and `cons_conf_idx` show very similar distributions for both classes, indicating that they may be weak stand-alone predictors.

- **nr_employed Mirrors the Economic Cycle**  
  Changes in `nr_employed` are consistent with the patterns seen in `euribor3m` and `emp_var_rate`, showing that some redundancy might be intriduced if these variables are used together.

**Conclusion:**  
Economic indicators, especially `euribor3m`, `emp_var_rate`, and `nr_employed`, provide **macro-context** rather than strong individual discrimination. Combined with other features, they help capture the economic phase in which campaigns occur.

### 8.2 Success Rates Across Categorical Variables

Here we compare subscription success rates across categorical features such as job, marital status, education, loans, contact type, month, day of week, and previous campaign outcome.

**Key observations:**

- **Job:**  
  Students and retired individuals show some of the highest campaign success rates, while blue-collar and certain services roles are less responsive.

- **Education:**  
  Higher education levels tend to correlate with slightly higher subscription likelihood.

- **Contact Method:**  
  Cellular contact has much higher conversion rates than telephone, indicating a clear preference.

- **Month (Seasonality):**  
  Subscription success is not uniform over the year: months like **March, September, October, and December** perform better, while **May, June, July and August** are weaker.

- **Previous Campaign Outcome (`poutcome`):**  
  Customers with a previously successful outcome are much more likely to subscribe again.

- **Other Categorical Features:**  
  Variables like marital status, housing loan, and personal loan show weaker stand-alone effects.

**Conclusion:**  
The most informative categorical features are **job**, **education**, **month**, **contact**, and **poutcome**. These will be central for modeling.

### 8.3 Numerical Feature Distributions

The histograms of numerical variables (age, campaign, duration, pdays, previous, economic indicators) highlight strong skewness and long tails.

**Key observations:**

- **Age:**  
  Right-skewed, concentrated roughly between 30–50 years.

- **Campaign:**  
  Highly right-skewed; most customers are contacted only a few times, with a long tail of repeated attempts.

- **pdays:**  
  Dominated by the value 999 (no previous contact), effectively making it close to a binary indicator.

- **Previous:**  
  Mostly zero, with a small subset of customers having prior contacts.

- **Economic Features:**  
  Show discrete clusters because they change at fixed time intervals (monthly/quarterly).

**Conclusion:**  
Many numeric variables are **heavily skewed** and contain outliers. This reinforces the need for:
- Careful treatment in linear models (e.g., transformations or binning)
- Robustness from tree-based models that are able to handle skewed distributions naturally.

### 8.4 Numerical Variables vs Subscription Outcome

Boxplots of numeric features split by target (`y`) show how the distribution changes between subscribed and non-subscribed customers.

**Key observations:**

- **Campaign:**  
  Successful subscriptions tend to happen with **fewer contact attempts**, indicating diminishing returns from repeated calls.

- **pdays:**  
  Customers contacted in previous campaigns (pdays < 999) show higher subscription rates than those never contacted.

- **Economic Variables:**  
  Subscription events are more frequent in periods with **lower `euribor3m`** and **negative `emp_var_rate`**, consistent with the macro observations.

- **Age:**  
  Subscribed customers tend to be slightly older on average.

**Conclusion:**  
Key numeric predictors visually include **campaign**, **pdays**, and **selected economic indicators**, with **age** also providing moderate signal.

### 8.5 Pearson Correlation – Numerical Variables

The Pearson correlation heatmap highlights correlations among numerical variables, with |r| > 0.7 occurrences framed for emphasis.

**Key observations:**

- **Strong multicollinearity among economic indicators:**  
  `euribor3m`, `emp_var_rate`, `nr_employed`, and related variables are highly correlated.

- **Weak direct correlation with target:**  
  No single numeric feature has a high linear correlation with `y`.

- **Relationships among campaign variables:**  
  Some moderate relationships appear (e.g., between `pdays` and `previous`), but not enough to cause major concern.

**Conclusion:**  
Multicollinearity should be handled carefully in **linear** models (via regularization). Tree-based models are naturally more robust to it.

### 8.6 Cramér’s V – Categorical Feature Associations

Cramér’s V is used to measure association strength between pairs of categorical variables.

**Key observations:**

- **Strong associations:**
  - `housing` ↔ `loan` – customers with one type of loan are more likely to have another  
  - `contact` ↔ `month` – operational choices around when and how customers are contacted

- **Moderate associations:**
  - `job` ↔ `education` – expected, as job and education level are naturally related

- **Associations with target:**
  - `poutcome` ↔ `y` – previous success is a strong signal for current success  
  - `month` ↔ `y` – confirms the importance of seasonality

**Conclusion:**  
Categorical multicollinearity is manageable, and several features (especially `poutcome` and `month`) carry strong predictive signals.

### 8.7 Pairplots – Economic Indicators & Target

The pairplots of economic indicators colored by `y_bin` show how economic conditions evolve with respect to the outcome.

**Key observations:**

- Clustered patterns due to **discrete** time steps in economic variables
- Subscriptions are somewhat more frequent during periods with:
  - Lower `euribor3m`
  - Negative `emp_var_rate`
  - Lower `nr_employed`

**Conclusion:**  
Economic variables act more as **contextual timing features** than as strong discriminators on their own, but they are still useful when combined with campaign and customer information.

### 8.8 Pairplot – Separation by Target

The pairplot focused on key numeric variables and the target confirms:

- **Campaign attempts:**  
  Fewer attempts are associated with higher success.

- **Previous contact history (`pdays`):**  
  Prior contacts correlate with higher subscription probability.

- **Economic conditions:**  
  Successes are clustered in less favorable economic periods (for the economy), which is coherent with earlier findings.

**Conclusion:**  
The combination of **campaign behavior**, **previous contact history**, and **economic timing** should be particularly important for modeling.

### 8.9 Mixed PairGrid with Regression Lines

The mixed PairGrid with regression lines and KDE plots shows:

- Strong linear structure among economic indicators (as expected)
- Weak linear structure among other features like age, campaign, and pdays
- Heavy overlap between subscribed and non-subscribed classes

**Conclusion:**  
Relationships in the dataset are largely **nonlinear**, indicating that tree-based models (e.g., XGBoost) might perform better over purely linear approaches.

## 9. Exploratory Data Analysis Summary

The EDA reveals several key patterns that will guide modeling and feature engineering.

### 9.1 Demographics & Customer Profiles

- **Age:**  
  Older customers, particularly seniors, show higher subscription rates, while middle-aged customers are less responsive.

- **Job & Education:**  
  Students and retirees stand out with above-average subscription rates, and higher education is correlated with slightly better conversion.

### 9.2 Contact & Campaign Dynamics

- **Number of contacts (`campaign`):**  
  Successful outcomes usually occur with a **small number of contact attempts**. Aggressive repeated contacting shows diminishing returns.

- **Previous contact history (`pdays`, `previous`, `poutcome`):**  
  Customers previously contacted, especially those with **prior success**, are more likely to subscribe again.

- **Contact type (`contact`):**  
  Cellular calls perform substantially better than telephone calls.

### 9.3 Seasonality & Economic Context

- **Month:**  
  Subscription rates vary significantly across the year, with certain months (e.g., March, September, October, December) performing much better.

- **Economic indicators:**  
  Lower interest rates and weaker employment conditions appear correlated with higher subscription rates, suggesting that uncertain times increase demand for term deposits.

### 9.4 Target Distribution & Class Imbalance

- The dataset is **highly imbalanced** (~11% positive class).
- No single variable provides perfect separation; predictive power comes from **combinations** of demographic, campaign, and economic features.

**Overall conclusion:**  
The most influential dimensions are **campaign intensity**, **previous contact history**, **timing (month + economic context)**, and **customer segment** (age, job, education). These will be central in the modeling phase.

## 10. Feature Engineering & Modeling Recommendations

The EDA revealed several strong patterns in customer behavior, campaign dynamics, and economic context.  
This section summarizes concrete feature engineering steps and modeling strategies inspired by those findings.


### 10.1 Handle Skewed Numerical Variables

Many numerical variables (such as `campaign`, `pdays`, `previous`, and `duration`) show extreme skewness and long tails.

To address this:

- **Binning**
  - `pdays`: Convert to a binary indicator  
    `was_previously_contacted = (pdays != 999)`
  - `campaign`: Group into low / medium / high contact intensity  

- **Log Transformations (optional for linear models)**
  - `campaign`, `previous`, `age` (depending on distribution)

Has previously mentioned, tree-based models handle skew naturally, so transformations are most important for models requiring linear assumptions.


### 10.2 Encode Categorical Variables

Categorical variables such as job, education, contact type, and month must be encoded before modeling.

Recommended encoding strategy:

- **One-hot encoding** for general use
- **Target encoding** or category grouping for high-cardinality variables
- Treat `"unknown"` as a meaningful level

These operations ensure robustness for both linear and nonlinear models.


### 10.3 Interaction Features

EDA revealed several nonlinear interactions worth considering in the following phases of the project:

- `campaign` × `previous`  
- `month` × economic indicators
- `job` × `age`  

Tree-based models naturally learn these, but explicit interactions can significantly improve linear models like Logistic Regression. The features should be used carefully as they might introduce redundancy and afect model performance.


### 10.4 Seasonality Features

Given strong monthly patterns, incorporating seasonality on model definition shows a strong performance indication.

Recommended features:

- **Cyclic encodings** (`sin(month)`, `cos(month)`)
- **Quarter of the year**
- Seasonal flags such as:
  - `is_summer`
  - `is_year_end`
  - `is_campaign_peak`

These operations help models learn temporal behaviors that affect subscription likelihood.


### 10.5 Remove Leakage & Low-Value Features

- **Drop `duration`**  
  This variable leaks target information because it is only known *after* the call.  
  Including it leads to unrealistic performance.

- Consider removing or down-weighting low-value features:
  - `marital`
  - `housing`
  - `loan`  
  These showed weak stand-alone predictive power in the EDA.

By eliminating low-value features we can simplify models, reduce noise and improve performance.


### 10.6 Scaling

Scaling is required only for models based on distance or linearity:

- **Scale these variables** for Baseline Models and Interpretable Linear Model:
  - `age`  
  - `campaign`  
  - `pdays`  
  - economic indicators  

- **No scaling required** for Random Forest, XGBoost, or other tree-based methods.


### 10.7 Model Selection Strategy

As per usual, a modeling approach should consider the testing of more than one model, starting firts with simpler more direct models to evaluate performance and the need to move to more complex ones. The following progression should be considered in the next phase of the project:

#### **1. Baseline Models**
- Dummy Classifier (predicts the majority class)
- Basic Logistic Regression  

These provide a performance reference point and help verify that more complex models are learning meaningful patterns.

#### **2. Interpretable Linear Model**
- **Logistic Regression with regularization**  
  - Highly interpretable coefficients  
  - Useful for communicating feature effects  
  - Helps diagnose multicollinearity and linear separability  

However, due to nonlinear patterns and multicollinearity, Logistic Regression is unlikely to be the top performer.

#### **3. Tree-Based Models (Primary Modeling Phase)**

**Random Forest**
- Robust to skewness and outliers  
- Captures nonlinear interactions  
- Provides useful feature importance  

**XGBoost (Recommended Final Model)**
- Excels at handling imbalanced datasets  
- Learns complex nonlinear patterns  
- Handles high-dimensional one-hot encodings  
- Works well with mixed numerical + categorical engineered features  
- Compatible with SHAP for explainability  

### 10.8 Handling Class Imbalance

The target variable is heavily imbalanced (~11% “yes”), requiring deliberate mitigation strategies.

Recommended approaches:

- **Class Weights**
  - For Logistic Regression  
  - For XGBoost via `scale_pos_weight = negatives / positives`  
    (approx. 8 in this dataset)

- **Oversampling (SMOTE / ADASYN)**
  - Useful for linear models  
  - Less necessary for tree-based models but worth testing

- **Undersampling**
  - Can be used for baseline comparison  
  - Not recommended for final models due to loss of information

Class weighting is the preferred approach for tree-based models.


### 10.9 Evaluation Metrics

Accuracy alone should not be considered as a model performance evaluation metric, as it is misleading when dealing with imbalanced classes.  
With this in mind, more informative metrics should be considered, such as:

- **F1-score (Primary Metric)**  
  Balances precision and recall for minority-class detection.

- **Precision**  
  Controls the number of unnecessary calls to uninterested customers.

- **Recall**  
  Measures how many actual subscribers the model identifies — essential for business goals.

- **ROC-AUC**  
  Evaluates class separation across thresholds; good for global model comparison.

- **PR-AUC**  
  More reflective of model performance under imbalance.

- **Confusion Matrix**  
  Crucial for understanding operational trade-offs:
  - false positives → wasted calls  
  - false negatives → missed opportunities  


### **Final Recommendation**

Based on all EDA findings and model considerations **XGBoost is recommended as the final model**. Its performance, robustness to skewed data, handling of nonlinear relationships, and compatibility with SHAP for explainability make it the best candidate for the model to be used on this project. However it should be supported by a well-designed feature engineering pipeline, careful class weighting, and further evaluation of the results.